# Historical scenario replay

**Prerequisites:** complete `01_foundations/market_data_and_curves.ipynb`, `05_portfolio/portfolio_construction_and_valuation.ipynb`, and `06_scenarios/scenarios_and_stress_testing.ipynb` before this notebook. You should be comfortable with `MarketContext` JSON, portfolio specs, and the scenario framework.

## Concepts: historical replay vs deterministic stress

The scenario framework applies **deterministic bumps** (parallel shifts, node shocks, equity moves) to the current market state. Even the built-in historical templates (GFC 2008, COVID 2020) are **pre-packaged bump magnitudes**, not actual market data.

**Historical replay** takes a different approach: you supply a sequence of **dated market snapshots** (real curves, spreads, vol surfaces from a time series database) and the engine steps a **static portfolio** through each date, producing:

- **PvOnly** -- portfolio value at each date
- **PvAndPnl** -- daily and cumulative P&L
- **FullAttribution** -- factor decomposition (rates, credit, FX, vol, etc.) at each step

The engine is data-source agnostic -- it accepts `[{"date": ..., "market": ...}]` and doesn't care where the snapshots came from.

## Setup: portfolio and market snapshots

We build a single USD deposit and create five dated market snapshots tracing a
short rate path (100bp → 125bp → 200bp → 150bp → 175bp over five days).

Unlike the other notebooks in this section, the market here is **not**
`_shared.build_demo_market()`. Replay needs a *sequence* of dated snapshots that
differ from one another, and the shared demo market is a single snapshot — so
each step's curve is built explicitly to drive the rate path. In production you
would load these snapshots from a time-series database instead.

Portfolio values are read through **`PortfolioValuation`**, the typed wrapper,
rather than by indexing into `total_base_ccy` on the raw step payload.

In [1]:
import json
from datetime import date, timedelta

import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF
from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.portfolio import PortfolioValuation, replay_portfolio

AS_OF = DEMO_AS_OF.isoformat()

def step_total(step: dict) -> float:
    """Base-currency portfolio value for one replay step, via the typed wrapper."""
    return PortfolioValuation.from_json(json.dumps(step["valuation"])).total_value

# --- Portfolio: one USD deposit ---
portfolio_json = json.dumps(
    {
        "id": "replay-demo",
        "as_of": AS_OF,
        "base_ccy": "USD",
        "entities": {"DESK": {"id": "DESK"}},
        "positions": [
            {
                "position_id": "P1",
                "entity_id": "DESK",
                "instrument_id": "DEP-6M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-6M",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": AS_OF,
                        "maturity": "2025-07-15",
                        "day_count": "Act360",
                        "quote_rate": 0.05,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            }
        ],
    }
)

# --- Market snapshots: 5 days along a rate path ---
def make_market(as_of: date, rate_bp: float) -> str:
    """Build a MarketContext JSON with a USD-OIS curve at the given rate."""
    rate = rate_bp / 10_000.0
    mc = MarketContext()
    mc.insert(DiscountCurve.flat("USD-OIS", as_of, rate))
    return mc.to_json()

base_date = DEMO_AS_OF
rate_path_bp = [100.0, 125.0, 200.0, 150.0, 175.0]  # rates rise, dip, then rise

snapshots = []
for i, bp in enumerate(rate_path_bp):
    d = base_date + timedelta(days=i)
    snapshots.append({"date": d.isoformat(), "market": json.loads(make_market(d, bp))})

print(f"Created {len(snapshots)} snapshots:")
for snapshot, bp in zip(snapshots, rate_path_bp):
    print(f"  {snapshot['date']}  flat USD-OIS @ {bp:.0f}bp")


Created 5 snapshots:
  2025-01-15  flat USD-OIS @ 100bp
  2025-01-16  flat USD-OIS @ 125bp
  2025-01-17  flat USD-OIS @ 200bp
  2025-01-18  flat USD-OIS @ 150bp
  2025-01-19  flat USD-OIS @ 175bp


## PvOnly mode

The simplest mode: just the portfolio value at each date. No P&L or attribution computation.

In [2]:
config_pv = json.dumps({"mode": "PvOnly"})
snapshots_json = json.dumps(snapshots)

result_json = replay_portfolio(portfolio_json, snapshots_json, config_pv)
result = json.loads(result_json)

print(f"{'Date':<14} {'Portfolio Value':>16}")
print("-" * 32)
for step in result["steps"]:
    print(f"{step['date']:<14} {step_total(step):>16,.2f}")

Date            Portfolio Value
--------------------------------
2025-01-15         1,019,997.65
2025-01-16         1,018,751.75
2025-01-17         1,014,994.97
2025-01-18         1,017,563.90
2025-01-19         1,016,356.26


## PvAndPnl mode

Adds daily and cumulative P&L. Step 0 is the anchor (no P&L computed).

In [3]:
config_pnl = json.dumps({"mode": "PvAndPnl"})

result_json = replay_portfolio(portfolio_json, snapshots_json, config_pnl)
result = json.loads(result_json)

print(f"{'Date':<14} {'PV':>14} {'Daily P&L':>14} {'Cumul P&L':>14}")
print("-" * 58)
for step in result["steps"]:
    pv = step_total(step)
    daily = float(step["daily_pnl"]["amount"]) if step["daily_pnl"] else None
    cumul = float(step["cumulative_pnl"]["amount"]) if step["cumulative_pnl"] else None
    daily_str = f"{daily:>14,.2f}" if daily is not None else f"{'--':>14}"
    cumul_str = f"{cumul:>14,.2f}" if cumul is not None else f"{'--':>14}"
    print(f"{step['date']:<14} {pv:>14,.2f} {daily_str} {cumul_str}")

Date                       PV      Daily P&L      Cumul P&L
----------------------------------------------------------
2025-01-15       1,019,997.65             --             --
2025-01-16       1,018,751.75      -1,245.90      -1,245.90
2025-01-17       1,014,994.97      -3,756.78      -5,002.68
2025-01-18       1,017,563.90       2,568.93      -2,433.75
2025-01-19       1,016,356.26      -1,207.64      -3,641.39


## Takeaways statistics

The replay result includes aggregate statistics: total P&L, max drawdown, and the peak/trough dates.

In [4]:
s = result["summary"]
print(f"Start date:     {s['start_date']}")
print(f"End date:       {s['end_date']}")
print(f"Num steps:      {s['num_steps']}")
print(f"Start value:    {float(s['start_value']['amount']):>14,.2f}")
print(f"End value:      {float(s['end_value']['amount']):>14,.2f}")
print(f"Total P&L:      {float(s['total_pnl']['amount']):>14,.2f}")
print(f"Max drawdown:   {float(s['max_drawdown']['amount']):>14,.2f}")
print(f"Max DD %:       {s['max_drawdown_pct']:>13.2%}")
print(f"DD peak date:   {s['max_drawdown_peak_date']}")
print(f"DD trough date: {s['max_drawdown_trough_date']}")

Start date:     2025-01-15
End date:       2025-01-19
Num steps:      5
Start value:      1,019,997.65
End value:        1,016,356.26
Total P&L:           -3,641.39
Max drawdown:         5,002.68
Max DD %:               0.49%
DD peak date:   2025-01-15
DD trough date: 2025-01-17


## FullAttribution mode

The richest mode: at each step, the engine runs `attribute_portfolio_pnl` to decompose P&L into factors (carry, rates, credit, FX, vol, etc.). This reuses the same attribution infrastructure as single-step scenarios.

In [5]:
config_attr = json.dumps(
    {
        "mode": "FullAttribution",
        "attribution_method": "Parallel",
    }
)

result_json = replay_portfolio(portfolio_json, snapshots_json, config_attr)
result = json.loads(result_json)

print(f"{'Date':<14} {'Daily P&L':>12} {'Carry':>12} {'Rates':>12} {'Residual':>12}")
print("-" * 64)
for step in result["steps"]:
    d = step["date"]
    if step["attribution"] is None:
        print(f"{d:<14} {'(anchor)':>12} {'--':>12} {'--':>12} {'--':>12}")
        continue
    attr = step["attribution"]
    daily = float(step["daily_pnl"]["amount"])
    carry = float(attr["carry"]["amount"])
    rates = float(attr["rates_curves_pnl"]["amount"])
    resid = float(attr["residual"]["amount"])
    print(f"{d:<14} {daily:>12,.2f} {carry:>12,.2f} {rates:>12,.2f} {resid:>12,.2f}")

Date              Daily P&L        Carry        Rates     Residual
----------------------------------------------------------------
2025-01-15         (anchor)           --           --           --
2025-01-16        -1,245.90        28.33    -1,274.24         0.00
2025-01-17        -3,756.78        35.37    -3,792.15         0.00
2025-01-18         2,568.93        56.39     2,512.54         0.00
2025-01-19        -1,207.64        42.40    -1,250.04         0.00


## Takeaways

- **`replay_portfolio`** accepts a portfolio spec, a JSON array of dated market snapshots, and a config selecting the replay mode.
- **Three modes**: `PvOnly` (fast scan), `PvAndPnl` (daily/cumulative P&L), `FullAttribution` (factor decomposition).
- The engine is **data-source agnostic** -- you supply `[{"date": ..., "market": ...}]` from any source (database, files, API).
- **Summary statistics** include total P&L, max drawdown amount/percentage, and peak/trough dates.
- Attribution reuses the existing `attribute_portfolio_pnl` infrastructure -- same parallel/waterfall/Taylor methods available.
- The **portfolio is static** throughout the replay, isolating pure market P&L from trading activity.

**Next:** combine replay with your own market data pipeline to answer questions like *"what would my portfolio P&L have been through March 2020?"*